## 进程 vs 程序 + PCB（jos struct Env 对照）

> 2026-10-12 · 来源：学习对话（进程模块块1：定义与抽象）

- **我的理解**：从 OS 模块块1（特权指令/trapframe）直进进程模块。进程概念与已吃透的 jos env/trapframe 直接衔接。
- **知识点**：
  - **进程 = 程序 + 数据 + PCB**，是程序的一次执行过程。程序是静态的指令集合，进程是动态的、有生命周期的。
  - **进程与程序不是一一对应**：一个程序可对应多个进程（开三个终端跑同一个 bash）；一个进程执行中也可涉及多个程序（exec 换程序，PID 不变）。
  - **PCB 是进程存在的唯一标志**。内容四类：①标识信息（PID/父进程）②处理机状态（寄存器现场——就是前面讲的 trapframe）③调度信息（状态/优先级）④控制信息（内存指针、资源清单、I/O 状态）。
  - 程序和数据可以被多个进程**共享**，PCB 不能共享（一人一份档案）。
- **项目映射**（jos，实读核实）：
  - `jos/inc/env.h` 的 `struct Env` 就是 PCB 的教学版：`env_tf`（Trapframe=处理机状态现场）｜`env_id` + `env_parent_id`（标识）｜`env_status`（状态）｜`env_pgdir`（内存指针/地址空间）｜`env_pgfault_upcall` + IPC 字段（控制/通信信息）——**教材 PCB 四类内容逐项对得上**。
  - 上一专题的 trapframe 现在看到了归宿：它就嵌在 PCB 里，是「处理机状态」那个字段。进程 = 地址空间（env_pgdir）+ 现场（env_tf）+ 档案（其余字段）。
- **考试侧重点**：★核心题型 3「PCB 内容判断」——判据：PCB 放的是**关于**程序的信息与指针，不放程序代码本身。「指令序列属于 PCB」必错。
- **易错点**：进程与程序一一对应（错）；PCB 里存指令序列（错，存的是指针/状态）；PCB 可以共享（错）。
- **待解决问题**：待补充
- **复习计划**：R1 @ 2026-10-13


## 进程五态与状态转换（含挂起；jos 状态映射）

> 2026-10-12 · 来源：学习对话（进程模块块1）

- **我的理解**：待补充
- **知识点**：
  - **五态**：新建 → 就绪 → 运行 → 阻塞 → 终止。四条主边，每条记「谁触发」：就绪→运行=**被调度**；运行→就绪=**时间片到/被抢占**；运行→阻塞=**主动等事件**（I/O 请求）；阻塞→就绪=**事件完成**（中断）。
  - **两条铁边**：①阻塞**不能**直达运行（必须先进就绪排队）②时间片到是运行→**就绪**（不是阻塞——它不缺事件，只缺 CPU）。
  - **区分就绪/阻塞的一句话**：缺的是 CPU 还是事件？调度器只从就绪队列里挑人；阻塞的进程给 CPU 也没用。
  - **挂起（七态）**：挂起 = **被换出到外存**（中级调度，内存不够时）。活跃就绪↔静止就绪、活跃阻塞↔静止阻塞。挂起 ≠ 阻塞：阻塞在等事件且**仍在内存**；挂起是整体被换出，与等不等事件正交。
- **项目映射**（jos，实读核实）：
  - `jos/inc/env.h` 状态枚举只有五个：`ENV_FREE`/`ENV_DYING`/`ENV_RUNNABLE`/`ENV_RUNNING`/`ENV_NOT_RUNNABLE`，映射：RUNNABLE≈就绪、RUNNING≈运行、NOT_RUNNABLE≈阻塞、DYING/FREE≈终止。**没有独立的「新建」态**。
  - 妙处：`sys_exofork`（kern/syscall.c，实读）建出的子进程**初始即 ENV_NOT_RUNNABLE**，等父进程 `sys_env_set_status` 才转 RUNNABLE——jos 用同一个状态位兼了「新建」和「阻塞」两个语义。**印证模块边界标注：真实内核状态名与教材五态不一一对应，408 判卷只认教材术语**。
  - 阻塞的真身：NOT_RUNNABLE 多发生在等 IPC（`env_ipc_recving`）；被唤醒 = 发送方把它置回 RUNNABLE——就是「事件完成→阻塞→就绪」那条边的代码版。
- **考试侧重点**：★核心题型 1「状态转换触发条件」——逐事件对边；☆题型 2「含挂起的状态数」——换出即状态翻倍。
- **易错点**：阻塞→运行直连（不存在）；时间片到算阻塞（错）；「挂起 = 阻塞的一种」（错，两根轴）；就绪→阻塞（不存在，阻塞只能从运行主动进入）。
- **待解决问题**：待补充
- **复习计划**：R1 @ 2026-10-13


## 线程 vs 进程：分配单位 vs 调度单位（jos 无线程 / Linux 统一模型）

> 2026-10-12 · 来源：学习对话（进程模块块1）

- **我的理解**：待补充
- **知识点**：
  - **一句话定位**：进程是**资源分配**单位（地址空间、文件、设备都记在进程账上），线程是 **CPU 调度**单位（真正被排上处理机跑的执行流）。
  - **线程私有**：栈、寄存器现场（含 PC）、线程控制块 TCB、线程 ID。**线程共享**（同进程内）：地址空间（代码/数据/堆）、全局变量、打开文件表、信号处理。
  - **为什么引入线程**：同进程线程切换**不用换地址空间**（不 lcr3、不冲 TLB），开销小；共享数据不用走 IPC。对照 jos `env_run`：进程切换 = lcr3 + env_pop_tf；线程切换只剩 env_pop_tf 那半步。
  - **用户级 vs 内核级线程**：判据一条——**内核见不见得着线程**。用户级：线程表在用户态库，内核只看到整个进程 → 一个线程阻塞（陷入内核）整个进程全堵；内核级：内核逐一调度，一个堵其余照跑。映射：多对一（用户级）/ 一对一（内核级，Linux 专精此道）/ 多对多（组合）。
- **项目映射**：
  - jos：**无线程概念**，一个 env 就是一条执行流（一份 env_tf）。想并发？开两个 env，走 IPC——正好对应教材「进程级并发的代价」。
  - Linux：`task_struct` **统一**进程与线程，`clone()` 标志位决定共享哪些资源——共享地址空间的就是「线程」。边界：408 考教材的进程/线程二分，不考 clone 标志位。
- **考试侧重点**：★核心题型 4「进程 vs 线程辨析」——从分配单位/调度单位/共享性/开销四维作答；☆题型 5「用户级/内核级判定」——看阻塞是否拖垮整个进程。
- **易错点**：「线程是资源分配单位」（反了）；线程共享栈（错，栈必私有）；线程共享寄存器（错，现场必私有）；用户级线程一个阻塞其余还能跑（错，内核看不到线程只能整进程挂起）；「不引入线程就没有并发」（错，进程级也能并发，只是代价大）。
- **待解决问题**：待补充（调度章回来对号：线程如何改变调度对象）
- **复习计划**：R1 @ 2026-10-13


## fork 与进程创建：返回两次的物理实现（jos sys_exofork）

> 2026-10-13 · 来源：学习对话（进程模块块2：创建与控制）

- **我的理解**：待补充
- **知识点**：
  - **fork()**：创建子进程，复制父进程。**调用一次，返回两次**：父进程得子 PID，子进程得 0，失败得 −1。父子并发执行，先后不确定。
  - **子进程从 fork 返回处继续执行**（不是从头跑程序）——因为复制的是现场（trapframe），PC 指向的就是 fork 的返回点。
  - **「返回 0」不是魔法**：返回值走寄存器，子进程的返回值 = 把复制来的现场里 eax 改成 0。父进程 eax = 子 PID。
  - **输出次数**：连续 n 次 fork → 2ⁿ 个进程；`if(fork()==0) fork();` → 3 个（只有子再 fork）。数法：画进程树，每次 fork 所有活着的进程各裂变一个。
  - **printf 缓冲坑**：缓冲区在地址空间里，会被一并复制——无换行的 printf 输出可能在父子各出现一次（缓冲未刷新就被复制）。
- **项目映射**（jos，实读核实）：
  - `kern/syscall.c sys_exofork`：① `env_alloc` 建 PCB ② `env_status = ENV_NOT_RUNNABLE`（先冻结）③ `env_tf = curenv->env_tf`（现场整包复制）④ `env_tf.tf_regs.reg_eax = 0`（**子进程返回 0 就这一行**）。
  - `user/dumbfork.c` 注释原话：*the child will appear to have called sys_exofork() too — except that in the child, this "fake" call will return 0*。
  - dumbfork 流程 = fork 的手动版：exofork → 父进程逐页 `duppage` 复制地址空间 → `sys_env_set_status(envid, ENV_RUNNABLE)` 放行——**教材「新建→就绪：创建完成」那条边的代码版**。
- **考试侧重点**：★核心题型 1「fork 输出次数」、题型 2「父子继承判断」——继承：文件描述符表/环境变量/信号处理；不继承：PID、返回值。
- **易错点**：「fork 调用两次」（错，调用一次返回两次）；「子进程从程序开头执行」（错，从 fork 返回处）；数进程忘了 if 分支只让子 fork。
- **待解决问题**：待补充
- **复习计划**：R1 @ 2026-10-14


## exec / wait / 僵尸与孤儿（jos ENV_DYING 对照）

> 2026-10-13 · 来源：学习对话（进程模块块2）

- **我的理解**：待补充
- **知识点**：
  - **exec()**：用新程序**替换**当前进程的地址空间，**PID 不变**（进程没死，换了灵魂）。**成功不返回**——调用点所在的代码已被替换，后面是死代码；失败才返回 −1。
  - **wait()**：父进程等子进程结束并取回退出状态、回收残留。没有 wait 的子进程死亡 → **僵尸**：PCB 不能释放（退出状态没人收），占着系统表。
  - **僵尸 vs 孤儿**：僵尸 = 子先死、父没 wait（尸体等收）；孤儿 = 父先死、子活着（被 init 收养，init 会替它 wait，无害）。**僵尸杀不掉**（`kill -9` 无效——它已经死了，只能让父进程 wait 或父死掉转由 init 收尸）。
  - **fork+exec 分离的意义**：shell 能在两者之间插私货——重定向 fd（`>` `<`）、建管道（`|`）都是 fork 之后、exec 之前干的。合二为一（如 Windows CreateProcess）就没这个窗口。
- **项目映射**（jos，实读核实）：
  - jos 无 exec 系统调用；`lib/spawn.c spawn()` = fork+exec 的合体（建 env + 加载新程序），对照「分离 vs 合并」两种设计。
  - **ENV_DYING 是另一种僵尸**（`kern/env.c env_destroy`）：目标进程在别的 CPU 上 RUNNING 时不能就地释放 → 置 `ENV_DYING` 返回，注释：*A zombie environment will be freed the next time it traps to the kernel*——**死亡推迟到它下次陷入内核**。与教材僵尸（等 wait 收尸）机制不同，同名不同义，别混。
  - `kern/trap.c:368`：trap 进来发现 `env_status == ENV_DYING` 就地回收——「下次陷入内核」的兑现点。
- **考试侧重点**：☆题型 3「fork+exec+wait 组合」按语句画进程树；☆题型 5「僵尸/孤儿判定」看父子退出顺序与是否 wait。
- **易错点**：exec 新建进程（错，PID 不变）；exec 成功后执行后续语句（死代码）；僵尸可以 kill（错）；孤儿是错误状态（错，init 兜底）；把 jos 的 ENV_DYING 当教材僵尸。
- **待解决问题**：待补充
- **复习计划**：R1 @ 2026-10-14


## 写时复制 COW：error code W 位的闭环（dumbfork vs lib/fork.c）

> 2026-10-13 · 来源：学习对话（进程模块块2）

- **我的理解**：待补充（此前 error code 专题预埋的 COW 消费者，本 cell 兑现）
- **知识点**：
  - **COW（写时复制）**：fork 后父子**共享**物理页，页表项标成只读/COW；任一方**写**时触发缺页（P=1, W=1），内核查 error code 的 W 位 + 页引用计数 → 才真正复制一份私有的。读不复制。
  - **收益**：绝大多数页 fork 后根本不会被写 → 全复制是白干；COW 把复制成本摊到「真正发生写冲突」的那几页上。
  - **时间线闭环**（接 00 笔记 error code 专题）：fork 时把可写页标 COW → 子进程写共享页 → #PF（err = U1 W1 P1 = 0b111）→ pgfault handler 查 `err & FEC_WR`（写错误）→ 确认是 COW 陷阱 → 复制页、改回可写、返回重执行该指令（**fault 可重执行**的实例）。
- **项目映射**（jos，实读核实）：
  - **反例** `user/dumbfork.c`：eager 全复制——每页 `sys_page_alloc` + `sys_page_map` + `memmove` 三连，注释自嘲 *This is NOT what you should do in your fork*。教学用：先看贵的，才懂便宜的贵在哪。
  - **正例** `lib/fork.c duppage`：不复制，只 `sys_page_map` 共享映射；可写页（PTE_W）改标 `PTE_COW`；`PTE_SHARE` 页直接共享不保护。配套 pgfault upcall 按位分诊——**此前「COW 消费者在 lab4、尚无代码」的说法修正：代码在 `lib/fork.c`（此前查错了 user/ 目录），本仓已有**。
  - fork 成本对比：dumbfork 每页 3 次系统调用 + 全量内存搬移；COW fork 每页 1 次映射，写冲突页才额外复制。
- **考试侧重点**：☆题型 4「COW 计数」——只有**写**触发复制，读不算；一个页父子各写一次 → 最多复制 2 次（各得私有副本）。
- **易错点**：fork 时就复制全部页（这是 dumbfork 的错法/COW 的定义反例）；「COW 后父子永远共享」（错，写过一次就分家）；「读也触发复制」（错）；COW 缺页是 bug（错，是设计的正常慢路径）。
- **待解决问题**：待补充（内存管理章回来对号：缺页处理 → 请求调页 vs COW）
- **复习计划**：R1 @ 2026-10-14


## CSAPP shlab：异步回收的三个坑（信号不排队 / 竞争 / waitfg）

> 2026-10-22 · 来源：CSAPP 项目复习（`modules/408/CSAPP.md`） §7.6（实践仓 `learning-408/csapp/shlab/tsh.c`，目前是官方 starter）
**坑 1 · 信号不排队**：SIGCHLD 不排队，5 个子进程同时死可能只递送一次 → handler 里必须 `while (waitpid(-1, &st, WNOHANG | WUNTRACED) > 0)`，只 waitpid 一次会留下僵尸。

**坑 2 · fork 与 addjob 的竞争**：子进程可能在父进程 `addjob` 之前就退出并触发 handler 的 `deletejob`（删了一个还没加进去的 job）。解法：`fork` **前** `sigprocmask(SIG_BLOCK, SIGCHLD)`，`addjob` 后解除；且**子进程继承了阻塞掩码**，必须在 `execve` 前恢复。

**坑 3 · waitfg 不能再 waitpid 阻塞**：SIGCHLD 已被 handler 收走，`waitfg` 应循环 `sigsuspend(&empty)` 直到 jobs 表里没有该前台 job。

**另两条**：`setpgid(0,0)` 让子进程独立成组，`kill(-pid, sig)`（**负 pid = 发给进程组**）才能把 Ctrl-C/Ctrl-Z 转给整个前台作业；handler 里改了 `errno` 要还原。

**408 对应**：fork 返回两次、exec 后地址空间替换（pid 不变）、`wait/waitpid` 与 WNOHANG、僵尸进程、进程组与信号。


## goroutine vs std::thread：用户级线程与内核级线程的两张真实面孔（6.824 raft 双版本对照）

> 2026-10-22 · 来源：6.824 项目复习（`modules/408/6824.md` §7.2，实践仓 `learning-408/raft`：Go 版已过测 2A–2C，C++ 版 craft 为半成品骨架）

- **我的理解**：待补充
- **知识点**：
  - **同一个 raft，两套线程模型**：Go 版里一个节点随手 `go func` 就派生十几个并发单元（ticker / applier / 每对 peer 一个发 RPC 的 goroutine）；C++ 版却先造了一个**线程池**才敢开工——这不是风格差异，是两种线程模型的本质差异。
  - **goroutine = M:N 有栈协程**：初始栈仅 2 KB（按需增长），**阻塞不占内核线程**——Go 运行时把大量 goroutine 复用到少量 OS 线程（M）上（GMP 模型）。所以 raft.go 里「每个 RPC 一个 goroutine」是零心理负担的写法。
  - **std::thread = 1:1 内核线程**：每个线程是真实的内核资源（栈默认 8 MB 级、调度进内核）。随手开线程 = 内存与上下文切换成本爆炸，所以 C++ 版必须**线程池化**（预创建 + 任务队列复用），这正是实践仓里 threadpool 先于 raft 骨架被写完的原因。
  - **对照 408**：教材里「用户级线程 / 内核级线程」的差别是「内核感知不到用户级线程的存在」——goroutine 的阻塞靠**运行时**（用户态调度器）化解，内核只看到 M 个线程在跑；std::thread 的创建/阻塞/切换全部**陷入内核**。一个项目两版本 = 这组概念的天然实验对照组。
  - **临界区视角**：正因为 goroutine 便宜，Go 版敢用「一把大锁 rf.mu + 轮询 ticker」这种粗放并发结构；C++ 转换后线程是稀缺资源，架构被迫改成「asio 事件驱动 + 少量线程」——**线程模型的成本差异会反向塑造程序的架构**（这是双版本对照最值钱的洞察）。

- **易错点**：goroutine 便宜 ≠ 免费——6.824 官方特别提醒测试不回收 goroutine，泄漏的循环 goroutine 会吃光 CPU 让后续测试假失败（raft.go 的 `killed()` 检查就是为这个）。
- **联动**：锁与条件变量的对照（channel vs condition_variable）见 `12-同步与互斥.ipynb` 的 Raft cell。

## OS 的本质：硬件缺口的补丁（nand2tetris ch12 的两句话定义）

> 2026-10-22 · 来源：nand2tetris 项目复习（modules/408/NAND2TETRIS.md）
- Jack OS 的 8 个类**逐一对应一个硬件缺口**：无乘法器 → Math 库；无 MMU/malloc 指令 → Memory.alloc 堆管理；无 GPU → Screen 位图绘制；无字库 ROM → Output 字形；无中断 → Keyboard 轮询；无引导程序 → Sys.init
- 由此得到 OS 的两句话定义：**硬件抽象 + 资源管理**（把「硬件不会做的事」补上，把「有限的硬件」分好）
- 但 Jack OS **不是现代内核**：单程序、单地址空间、无并发、无隔离、无系统调用陷入 —— **没有「进程」这个抽象**
- 对照真实 OS：每一类缺口长大后都变成了子系统（调度器、文件系统、网络栈、驱动模型）
- 与本模块的连接：**「进程」补的是「并发 + 隔离」这两个缺口** —— Hack 的单程序时代不需要它；多道程序一出现，进程（与线程、调度、同步）就全被逼出来了


## 【修正】C++ craft 的真实线程模型：每 peer 一线程 + join，以及原子计数为什么必须用在这里

> 2026-10-24 · 来源：6.824 项目复习 v2 重核（`modules/408/6824.md` §7.3；实践仓 `learning-408/raft/craft/src/raft/node.cpp` 465 行，独立 git 仓）
> **本 cell 修正上一条 cell 的事实**：C++ 版 craft 不是「半成品骨架」，而是 1729 行、可编译、`node.cpp` 465 行的完整 raft 实现。下面这条更正让线程模型的对照更锋利。

- **我的理解**：待补充
- **知识点**：
  - **真实写法**：C++ 版 `startElection()` / `startHeartbeat()` 每次都是「对每个 peer 创建一个 `std::thread`，然后**逐个 join**」（node.cpp:291-321）。也就是**一轮广播 = 一批短命线程的创建 + 销毁**。Go 版则是「每 peer 一个 goroutine，发完就退出，调用方不等」。
  - **对照 408**：这正好演示了「内核级线程昂贵」的后果 —— 线程创建/回收要陷入内核，所以同一件事在 Go 里是零成本操作（`go func`），在 C++ 里就成了**架构级约束**（要么像这里一样接受每轮 N-1 次创建，要么上线程池）。**线程模型的成本会反向塑造程序结构**，这是教材里不会写、但代码里看得很清楚的一句。
  - **为什么这里能用 atomic 而 Go 版不用**：C++ 版用 `std::atomic<std::size_t> votes_granted{1}` + `std::atomic<bool> won_election` 的 `compare_exchange_strong`（node.cpp:287-308）来保证「谁先到多数派谁当选、只当选一次」。Go 版同样的计数只是普通局部变量 `voteCount` —— 因为它每次自增都在同一把 `rf.mu` 里。**两个版本都在保证同一件事，只是 Go 用锁、C++ 用原子操作**；判断依据仍然是「是否被同一把锁覆盖」，不是「变量是否共享」。
  - **顺带记一条会考的判据**：`atomic` 的 CAS 是「无锁也安全」的**乐观**路线，锁是「先占后干」的**悲观**路线。选哪个看临界区长短：这里只有一次自增（极短）→ 原子操作更合适；raft 的状态修改（长）→ 大锁更合适。

- **易错点**：`std::thread` 忘了 `join()` 或 `detach()` → 析构时 `std::terminate` 直接崩；Go 的 goroutine 漏了回收只是泄漏。C++ 的线程生命周期必须显式管，这也是为什么这里每个函数末尾都有一圈 `join`。
- **联动**：锁与生产者-消费者见 `12-同步与互斥.ipynb`；完整对照表（goroutine/channel/timer/RPC/序列化 八项）见 `modules/408/6824.md` §7.3。